# 03 — Hierarchical Clustering

Cluster brain regions by their mechanosensitive ion channel expression profiles.
This reveals which brain regions are most similar or different in their
mechanosensitive channel repertoire — relevant for predicting differential
FUS sensitivity across brain areas.

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

from src import gene_lists, data_loader, clustering

sns.set_theme(style='whitegrid', context='notebook')
%matplotlib inline

## 1. Load multi-donor averaged expression

In [ ]:
genes = list(gene_lists.ALL_MECHANOSENSITIVE.keys())
region_expr = data_loader.get_microarray_region_means(genes)

# Filter to regions with complete data
region_expr_clean = region_expr.dropna(axis=1, thresh=int(len(genes) * 0.7))
region_expr_clean = region_expr_clean.dropna(axis=0)
print(f"Clean matrix: {region_expr_clean.shape[0]} genes x {region_expr_clean.shape[1]} regions")

## 2. Full clustermap — all mechanosensitive genes

Dual-axis hierarchical clustering: brain regions (columns) and genes (rows).

In [ ]:
g = clustering.clustermap(
    region_expr_clean,
    title='All Mechanosensitive Channels — Brain Region Clustering',
    figsize=(22, 12),
    method='ward',
    z_score_rows=True,
)
plt.savefig('../figures/clustermap_all_channels.png', dpi=150, bbox_inches='tight')
plt.show()

## 3. FUS-core genes only

In [ ]:
core_genes = list(gene_lists.FUS_CORE_GENES.keys())
core_expr = region_expr_clean.loc[region_expr_clean.index.isin(core_genes)]

g_core = clustering.clustermap(
    core_expr,
    title='FUS Core Channels — Brain Region Clustering',
    figsize=(22, 8),
    method='ward',
    z_score_rows=True,
)
plt.savefig('../figures/clustermap_core_channels.png', dpi=150, bbox_inches='tight')
plt.show()

## 4. Region dendrogram with cluster cutoff

In [ ]:
fig, cluster_labels = clustering.region_dendrogram(
    region_expr_clean,
    method='ward',
    n_clusters=6,
    figsize=(20, 8),
    title='Brain Regions Clustered by Mechanosensitive Channel Expression (k=6)',
)
plt.savefig('../figures/dendrogram_regions.png', dpi=150, bbox_inches='tight')
plt.show()

if cluster_labels is not None:
    df_clean = region_expr_clean.dropna(axis=1)
    cluster_df = pd.DataFrame({
        'region': df_clean.columns,
        'cluster': cluster_labels
    })
    for c in sorted(cluster_df['cluster'].unique()):
        regions = cluster_df.loc[cluster_df['cluster'] == c, 'region'].tolist()
        print(f"\nCluster {c} ({len(regions)} regions): {', '.join(regions[:15])}{'...' if len(regions) > 15 else ''}")

## 5. Clustering by channel family

Do regions cluster differently when we consider only K2P, only TRP, or only Piezo channels?

In [ ]:
families = {
    'K2P (mechanosensitive)': list(gene_lists.K2P_MECHANOSENSITIVE.keys()),
    'TRP channels': list(gene_lists.TRP_GENES.keys()),
    'Piezo channels': list(gene_lists.PIEZO_GENES.keys()),
}

fig, axes = plt.subplots(3, 1, figsize=(20, 18))

for ax, (family_name, family_genes) in zip(axes, families.items()):
    fam_expr = region_expr_clean.loc[region_expr_clean.index.isin(family_genes)]
    if fam_expr.empty or fam_expr.shape[0] < 2:
        ax.text(0.5, 0.5, f'{family_name}: insufficient genes', ha='center', transform=ax.transAxes)
        continue

    fam_z = clustering.zscore_expression(fam_expr)
    fam_z.index = [gene_lists.get_display_name(g) for g in fam_z.index]

    sns.heatmap(
        fam_z, cmap='RdBu_r', center=0,
        xticklabels=True, yticklabels=True,
        ax=ax, linewidths=0.2,
        cbar_kws={'label': 'Z-score', 'shrink': 0.5}
    )
    ax.set_title(f'{family_name} — Regional Expression', fontsize=12)
    ax.tick_params(axis='x', rotation=90, labelsize=5)
    ax.tick_params(axis='y', labelsize=9)

plt.tight_layout()
plt.savefig('../figures/family_heatmaps.png', dpi=150, bbox_inches='tight')

## 6. Alternative distance metrics comparison

In [ ]:
for metric in ['euclidean', 'correlation']:
    for method in ['ward', 'average']:
        if metric == 'correlation' and method == 'ward':
            continue  # ward requires euclidean
        print(f"\n--- {method} linkage, {metric} distance ---")
        fig, labels = clustering.region_dendrogram(
            region_expr_clean,
            method=method,
            metric=metric,
            n_clusters=5,
            figsize=(18, 6),
            title=f'Region Dendrogram ({method}, {metric})',
        )
        plt.show()